# Colombia Open — Resultados Saber 11

Este notebook muestra cómo cargar el dataset **`educacion_saber11`** (resultados Saber 11 del ICFES, agregados por establecimiento y periodo, 2010-2022) con `colombia-open`, y cómo relacionarlos con el directorio de colegios (`educacion_establecimientos`).

> **Por qué por establecimiento:** agregar solo por municipio deja el resultado "suelto" — `codigo_dane_establecimiento` es la misma clave y el mismo nivel (institución) que usa `educacion_establecimientos`, así que el join entre los dos datasets es directo.

> **Nota (2026-10):** `colombia-open` está publicado en [PyPI](https://pypi.org/project/colombia-open/) — `pip install colombia-open` descarga y cachea el bundle de datos publicado en GitHub Releases automáticamente. Este dataset está disponible desde el bundle `v0.10.0`.

## Setup

In [ ]:
!pip install -q colombia-open

## Cargar el dataset

In [ ]:
import polars as pl
from colombia_open import ColombiaOpen

hub = ColombiaOpen()
saber11 = hub.load_polars("educacion_saber11")

print(saber11.shape)
saber11.head(10)

## Esquema

| Columna | Tipo | Descripción |
|---|---|---|
| `codigo_dane_establecimiento` | string (12 dígitos) | **Join key hacia `educacion_establecimientos`.** |
| `periodo` | string | Periodo de aplicación (`"20194"` = año 2019). |
| `anio` | int | Año derivado de `periodo`. |
| `cod_dpto` / `cod_mpio` | string (2 / 5 dígitos) | Ubicación del colegio — join key hacia `municipios`. |
| `naturaleza` | string, nullable | `"OFICIAL"` / `"NO OFICIAL"`. |
| `n_estudiantes` | int | Presentaciones agregadas en ese establecimiento-periodo. |
| `prom_matematicas` / `prom_ingles` | float | Disponibles en **todo** el rango 2010-2022. |
| `prom_lectura_critica` / `prom_sociales_ciudadanas` / `prom_c_naturales` / `prom_global` | float, nullable | **Nulos antes del periodo `20142`** (la prueba no los reportaba). |

Clave primaria: `(codigo_dane_establecimiento, periodo)`. Esquema completo: [`docs/datasets/educacion_saber11.md`](../../docs/datasets/educacion_saber11.md).

## Relacionar resultados con el directorio de establecimientos

In [ ]:
establecimientos = hub.load_polars("educacion_establecimientos")

saber11.filter(pl.col("anio") == 2022).join(
    establecimientos.filter(pl.col("anio") == 2022),
    on="codigo_dane_establecimiento",
    how="inner",
).select(
    ["nombre_establecimiento", "sector", "n_estudiantes", "prom_global"]
).sort("prom_global", descending=True).head(10)

## Brecha oficial vs. no oficial en matemáticas, por año

In [ ]:
(
    saber11.group_by(["anio", "naturaleza"])
    .agg(
        pl.col("prom_matematicas").mean().round(1).alias("prom_matematicas"),
        pl.col("n_estudiantes").sum().alias("n_estudiantes"),
    )
    .sort(["anio", "naturaleza"])
)

## Limitaciones conocidas

- **No es microdato** — es una agregación propia de `colombia-open` sobre el microdato público de ICFES (7.1M filas por estudiante).
- **`punt_ingles = -1` en la fuente** (periodos 2010-2013) es un centinela de "no administrado", no un puntaje real — ya se excluye del promedio antes de publicar.
- **El join hacia `educacion_establecimientos` hereda el ruido ya documentado** de ese dataset (algunos duplicados en `codigo_dane_establecimiento` en años "normales"); el defecto severo de 2023 de ese dataset no aplica aquí (Saber 11 solo llega hasta 2022).
- **Sin variables socioeconómicas** (estrato, educación de los padres...) — fuera de alcance de esta primera iteración.

Documentación completa: [`docs/datasets/educacion_saber11.md`](../../docs/datasets/educacion_saber11.md).